# Adversarial Search Strategies and Decision Trees
## PopOut — AI Assignment 2025/2026

**Group Members:** *(fill in your names here)*

---

### Overview

This notebook implements:

1. **PopOut game** — a Connect-4 variant with *drop* and *pop* moves.
2. **Monte Carlo Tree Search (MCTS) with UCT** — adversarial search algorithm.
3. **ID3 Decision Tree** — learned from two datasets:
   - Iris dataset (warm-up, numerical features requiring discretisation).
   - PopOut state dataset (generated via MCTS self-play).

All game scenarios are supported:
- Human vs. Human
- Human vs. Computer (MCTS)
- Computer vs. Computer (MCTS with different parameters / MCTS vs. Decision Tree)

> **Note on parameters:** The cells below use small `iterations` and `n_games` values for quick
> execution. For real experiments, increase these (e.g. `iterations=1000`, `n_games=50`).

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import random
import math
import time
from collections import Counter
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.game.board import Board, PLAYER1, PLAYER2, EMPTY, ROWS, COLS, SYMBOLS
from src.game.popout import PopOutGame
from src.game.player import HumanPlayer, MCTSPlayer, DecisionTreePlayer, RandomPlayer
from src.mcts.mcts import MCTS
from src.decision_tree.id3 import ID3DecisionTree
from src.datasets.generator import PopOutDatasetGenerator

print('All modules loaded successfully.')

---
## Part 1 — The PopOut Game

PopOut is a variant of Connect-4. Players alternate turns either:
- **Dropping** a disc into any non-full column (from the top), or
- **Popping** one of their own discs from the bottom of any column (shifts everything down).

The first to connect **four** discs horizontally, vertically, or diagonally wins.

Three special rules handle edge cases:
1. **Simultaneous four-in-rows** after a pop → the player who popped wins.
2. **Full board** → the mover may declare a draw.
3. **Three-fold repetition** → either player may declare a draw.

In [ ]:
def draw_board(board: Board, title: str = '', save_path: str = None):
    """Display the board graphically using matplotlib."""
    fig, ax = plt.subplots(figsize=(COLS * 0.8, ROWS * 0.8))
    ax.set_facecolor('#0a4fa0')
    colors = {EMPTY: 'white', PLAYER1: '#ffcc00', PLAYER2: '#ff4444'}

    for row in range(ROWS):
        for col in range(COLS):
            cell = board.grid[row][col]
            circle = plt.Circle((col + 0.5, ROWS - row - 0.5), 0.4,
                                 color=colors[cell], zorder=2)
            ax.add_patch(circle)

    ax.set_xlim(0, COLS)
    ax.set_ylim(0, ROWS)
    ax.set_xticks(range(COLS))
    ax.set_xticklabels([str(i + 1) for i in range(COLS)])
    ax.set_yticks([])
    ax.set_title(title, fontsize=12, fontweight='bold')
    legend = [
        mpatches.Patch(color='#ffcc00', label='X (Player 1)'),
        mpatches.Patch(color='#ff4444', label='O (Player 2)'),
    ]
    ax.legend(handles=legend, loc='upper right', fontsize=8)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=80)
    plt.show()
    plt.close()


# Demo board
demo_board = Board()
for col, player in [(3, PLAYER1), (3, PLAYER2), (4, PLAYER1),
                    (2, PLAYER2), (5, PLAYER1), (2, PLAYER1)]:
    demo_board.drop(col, player)
draw_board(demo_board, 'Demo board state')
print('Text representation:')
print(demo_board)

### 1.1 Human vs. Human

Run the cell below in an interactive session to play Human vs. Human.
Enter moves as prompted: move type (`drop` / `pop`) and column (1–7).

In [ ]:
# Uncomment to play Human vs Human (interactive session only)
# game = PopOutGame(HumanPlayer(), HumanPlayer())
# result = game.play(verbose=True)
print('(Human vs Human mode — uncomment the lines above to play)')

### 1.2 Human vs. Computer (MCTS)

The computer uses MCTS with the heuristic rollout policy.

In [ ]:
# Uncomment to play Human vs Computer (interactive session only)
# game = PopOutGame(HumanPlayer(), MCTSPlayer(iterations=500, rollout_policy='heuristic'))
# result = game.play(verbose=True)
print('(Human vs Computer mode — uncomment the lines above to play)')

### 1.3 Computer vs. Computer

We run games between two different AI agents to measure performance.

In [ ]:
def run_tournament(player1, player2, n_games: int = 5, verbose: bool = False):
    """Run n_games between player1 and player2 and return win counts."""
    results = {'player1': 0, 'player2': 0, 'draw': 0}
    for _ in range(n_games):
        game = PopOutGame(player1, player2)
        result = game.play(verbose=verbose)
        results[result] += 1
    return results


# Random vs Random (quick baseline)
print('Random vs Random — 5 games')
res = run_tournament(RandomPlayer(seed=1), RandomPlayer(seed=2), n_games=5)
print(f'  Player1 wins: {res["player1"]}  Player2 wins: {res["player2"]}  Draws: {res["draw"]}')

In [ ]:
# MCTS (random rollout) vs MCTS (heuristic rollout) — small iteration counts for quick demo
# Increase iterations for meaningful results (e.g. iterations=500, n_games=20)
N_GAMES = 5      # increase to 20+ for a real experiment
ITERS   = 50     # increase to 500+ for strong play

print(f'MCTS (random) vs MCTS (heuristic) — {N_GAMES} games, {ITERS} iterations each')
p1 = MCTSPlayer(iterations=ITERS, rollout_policy='random')
p2 = MCTSPlayer(iterations=ITERS, rollout_policy='heuristic')
results = run_tournament(p1, p2, n_games=N_GAMES)
print(f'  Player1 (random)    wins: {results["player1"]}')
print(f'  Player2 (heuristic) wins: {results["player2"]}')
print(f'  Draws:                    {results["draw"]}')

---
## Part 2 — Monte Carlo Tree Search (MCTS)

### 2.1 Algorithm description

MCTS repeatedly performs four steps:

1. **Selection** — starting from the root, choose children using UCT until reaching an unexpanded node:
$$\text{UCT}(v) = \frac{w_i}{n_i} + C \cdot \sqrt{\frac{\ln N_i}{n_i}}$$
   where $w_i$ = wins, $n_i$ = node visits, $N_i$ = parent visits, $C$ = exploration constant.

2. **Expansion** — add a child node for one untried move.

3. **Simulation** — play a random (or heuristic) rollout to a terminal state.

4. **Backpropagation** — update visit counts and win scores back up to the root.

The move leading to the **most-visited** child of the root is selected.

In [ ]:
# Demonstrate MCTS: block an immediate win
class _GameProxy:
    def __init__(self, board, player):
        self.board = board
        self.current_player = player
    def get_legal_moves(self):
        return self.board.get_legal_moves(self.current_player)


board = Board()
for col in range(3):
    board.drop(col, PLAYER2)   # PLAYER2 has 3 in a row

game_proxy = _GameProxy(board.copy(), PLAYER1)
mcts = MCTS(iterations=500, rollout_policy='heuristic')

t0 = time.time()
best_move = mcts.search(game_proxy)
elapsed = time.time() - t0

draw_board(board, 'Position: PLAYER2 has 3-in-a-row, PLAYER1 to move')
print(f'MCTS best move (500 iterations, heuristic): {best_move}  [{elapsed:.2f}s]')
print('Expected: ("drop", 3) — block the win at column 4')

In [ ]:
# Effect of iteration count on move quality
# We measure how often MCTS finds the correct blocking move

def blocking_accuracy(iterations: int, trials: int = 10, policy: str = 'random') -> float:
    """How often does MCTS pick the correct blocking move?"""
    correct = 0
    for seed in range(trials):
        b = Board()
        for col in range(3):
            b.drop(col, PLAYER2)
        proxy = _GameProxy(b.copy(), PLAYER1)
        m = MCTS(iterations=iterations, rollout_policy=policy).search(proxy)
        if m == ('drop', 3):
            correct += 1
    return correct / trials


iteration_counts = [10, 50, 100, 200, 500]
random_accs    = [blocking_accuracy(i, trials=5, policy='random')    for i in iteration_counts]
heuristic_accs = [blocking_accuracy(i, trials=5, policy='heuristic') for i in iteration_counts]

plt.figure(figsize=(7, 4))
plt.plot(iteration_counts, random_accs,    marker='o', label='random rollout')
plt.plot(iteration_counts, heuristic_accs, marker='s', label='heuristic rollout')
plt.xlabel('MCTS iterations')
plt.ylabel('Fraction correct (blocking move)')
plt.title('Move quality vs. MCTS iterations')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('mcts_iterations.png', dpi=80)
plt.show()
plt.close()
print('Plot saved to mcts_iterations.png')

### 2.2 Exploration constant analysis

We evaluate the impact of the UCT exploration constant $C$ on blocking accuracy.

In [ ]:
C_values = [0.5, 1.0, 1.414, 2.0, 3.0]
c_accs = []

for C in C_values:
    acc = 0
    trials = 5
    for _ in range(trials):
        b = Board()
        for col in range(3):
            b.drop(col, PLAYER2)
        proxy = _GameProxy(b.copy(), PLAYER1)
        m = MCTS(iterations=200, exploration_constant=C, rollout_policy='heuristic').search(proxy)
        if m == ('drop', 3):
            acc += 1
    c_accs.append(acc / trials)
    print(f'  C={C}: blocking accuracy = {acc / trials:.0%}')

plt.figure(figsize=(7, 4))
plt.plot(C_values, c_accs, marker='s', color='darkorange')
plt.xlabel('Exploration constant C')
plt.ylabel('Blocking accuracy (200 iters, heuristic)')
plt.title('Effect of UCT exploration constant')
plt.grid(True)
plt.tight_layout()
plt.savefig('mcts_exploration.png', dpi=80)
plt.show()
plt.close()
print('Plot saved to mcts_exploration.png')

---
## Part 3 — Decision Trees (ID3)

The ID3 procedure builds a decision tree by recursively selecting the attribute that maximises **information gain** (entropy reduction).

For continuous attributes we find the optimal split threshold automatically.

### 3.1 Dataset 1 — Iris (warm-up)

In [ ]:
# Load the iris dataset
gen = PopOutDatasetGenerator()
X_iris, y_iris, iris_features = gen.load_iris('data/iris.csv')

print(f'Iris dataset: {len(X_iris)} samples, {len(iris_features)} features')
print('Features:', iris_features)
print('Classes:', sorted(set(y_iris)))

df = pd.DataFrame(X_iris, columns=iris_features)
df['class'] = y_iris
df.describe()

In [ ]:
# Train / test split (80 / 20)
random.seed(42)
indices = list(range(len(X_iris)))
random.shuffle(indices)
split = int(0.8 * len(indices))

X_train = [X_iris[i] for i in indices[:split]]
y_train = [y_iris[i] for i in indices[:split]]
X_test  = [X_iris[i] for i in indices[split:]]
y_test  = [y_iris[i] for i in indices[split:]]

print(f'Train: {len(X_train)} samples  |  Test: {len(X_test)} samples')

In [ ]:
# Build the ID3 decision tree (all four features are continuous)
iris_tree = ID3DecisionTree(
    continuous_features=list(range(4)),
    feature_names=iris_features,
)
iris_tree.fit(X_train, y_train)

print('=== Iris Decision Tree ===')
iris_tree.display()

train_eval = iris_tree.evaluate(X_train, y_train)
test_eval  = iris_tree.evaluate(X_test,  y_test)
print(f'\nTrain accuracy: {train_eval["accuracy"]:.2%}  ({train_eval["correct"]}/{train_eval["total"]})')
print(f'Test  accuracy: {test_eval["accuracy"]:.2%}  ({test_eval["correct"]}/{test_eval["total"]})')

In [ ]:
# Effect of max_depth on Iris accuracy
depths = [1, 2, 3, 4, 5, None]
train_accs, test_accs = [], []

for d in depths:
    t = ID3DecisionTree(continuous_features=list(range(4)), max_depth=d)
    t.fit(X_train, y_train)
    train_accs.append(t.evaluate(X_train, y_train)['accuracy'])
    test_accs.append(t.evaluate(X_test,   y_test)['accuracy'])

depth_labels = [str(d) if d is not None else 'None' for d in depths]
x = range(len(depths))

plt.figure(figsize=(8, 4))
plt.plot(x, train_accs, marker='o', label='Train')
plt.plot(x, test_accs,  marker='s', label='Test')
plt.xticks(x, depth_labels)
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Iris — Effect of max_depth on ID3 accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('iris_depth.png', dpi=80)
plt.show()
plt.close()
print('Plot saved to iris_depth.png')

### 3.2 Dataset 2 — PopOut Game States

We generate a labelled dataset using MCTS self-play. Each sample is a board state (42 cells + 1 current player = 43 features). The label is the MCTS-recommended move as `movetype_col` (e.g. `drop_3`).

In [ ]:
# Generate PopOut dataset
# Increase n_games and mcts_iterations for a higher-quality dataset
print('Generating PopOut dataset (n_games=20, iterations=50)...')
dataset_gen = PopOutDatasetGenerator(mcts_iterations=50, rollout_policy='heuristic', seed=42)
X_popout, y_popout, popout_features = dataset_gen.generate(n_games=20)

print(f'Generated {len(X_popout)} (state, move) pairs')
print('\nTop 10 most frequent moves:')
for label, count in Counter(y_popout).most_common(10):
    print(f'  {label}: {count}')

In [ ]:
# Train / test split for PopOut dataset
random.seed(0)
idx = list(range(len(X_popout)))
random.shuffle(idx)
sp = int(0.8 * len(idx))

Xp_train = [X_popout[i] for i in idx[:sp]]
yp_train = [y_popout[i] for i in idx[:sp]]
Xp_test  = [X_popout[i] for i in idx[sp:]]
yp_test  = [y_popout[i] for i in idx[sp:]]

print(f'Train: {len(Xp_train)} | Test: {len(Xp_test)}')

In [ ]:
# Train the ID3 decision tree on the PopOut dataset
# Board cells are categorical (values: 0, 1, 2) → no continuous_features needed
popout_tree = ID3DecisionTree(
    continuous_features=[],
    feature_names=popout_features,
    max_depth=10,
)
popout_tree.fit(Xp_train, yp_train)

pte = popout_tree.evaluate(Xp_train, yp_train)
ppe = popout_tree.evaluate(Xp_test,  yp_test)
print(f'Train accuracy: {pte["accuracy"]:.2%}')
print(f'Test  accuracy: {ppe["accuracy"]:.2%}')

### 3.3 Computer vs. Computer — MCTS vs. Decision Tree Player

In [ ]:
dt_player   = DecisionTreePlayer(popout_tree, fallback=RandomPlayer(seed=7))
mcts_player = MCTSPlayer(iterations=50, rollout_policy='heuristic')

print('Decision Tree player vs. MCTS player (50 iters) — 5 games')
results = run_tournament(dt_player, mcts_player, n_games=5)
print(f'  Decision Tree wins: {results["player1"]}')
print(f'  MCTS wins:          {results["player2"]}')
print(f'  Draws:              {results["draw"]}')

---
## Part 4 — Discussion and Conclusions

*(Fill in your analysis, results discussion, and conclusions here.)*

### Key findings
1. **MCTS performance** increases with the number of iterations and saturates beyond a certain point.
2. The **heuristic rollout policy** outperforms random rollout, especially at low iteration counts.
3. The **exploration constant C** controls the explore/exploit trade-off; $\sqrt{2} \approx 1.414$ is a common default.
4. The **ID3 tree** on Iris achieves high accuracy (typically > 95%) while remaining interpretable.
5. The PopOut decision tree accuracy depends on dataset size and quality of MCTS labels.

### Constraints considered
- No scikit-learn for decision tree construction.
- MCTS uses the UCT formula for node selection.
- All three PopOut special rules are implemented.

### Future work
- Alpha-beta pruning as an alternative adversarial search baseline.
- Random forest ensemble to improve PopOut move prediction.
- Interactive GUI with pygame.